# UNet++ EfficientNet-B4 — Phát Hiện Vết Dầu Tràn Trên Ảnh SAR

| Thông tin | Chi tiết |
|-----------|----------|
| Model | UNet++ |
| Encoder | EfficientNet-B4 (pretrained ImageNet) |
| Dataset | `deep-sar-oil-spill-segmentation-refined` |
| Task | Binary Segmentation: background vs oil spill |
| Framework | PyTorch + segmentation_models_pytorch |
| Input size | `256 × 256` (resize từ ảnh gốc `416 × 416`) |
| GPU | Kaggle P100 (16 GB) |

---
**Cấu trúc Notebook (11 cell):**
1. Cài đặt thư viện và import
2. CONFIG tập trung
3. Fix seed
4. Load và EDA dataset
5. Pipeline tiền xử lý chung
6. Dataset class và DataLoader
7. Định nghĩa model UNet++
8. Training loop
9. Visualize quá trình train
10. Evaluation trên tập validation
11. Inference với API thống nhất

## Cell 1. Cài đặt thư viện và import

In [ ]:
!pip install -q segmentation-models-pytorch albumentations opencv-python-headless

import os
import csv
import json
import random
import shutil
from pathlib import Path
from typing import List, Optional, Tuple

import albumentations as A
import cv2
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import segmentation_models_pytorch as smp
import torch
import torch.nn as nn
from albumentations.pytorch import ToTensorV2
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from torch.cuda.amp import autocast, GradScaler

print(f'PyTorch   : {torch.__version__}')
print(f'SMP       : {smp.__version__}')
print(f'CUDA avail: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU       : {torch.cuda.get_device_name(0)}')
    print(f'VRAM      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print('\n✓ Tất cả thư viện đã được import thành công')

## Cell 2. CONFIG tập trung

> **Quy ước bắt buộc:** Mọi hyperparameter, đường dẫn và ngưỡng phân lớp đều phải đọc từ `CONFIG`. Không hardcode bất kỳ giá trị nào ở các cell sau.

In [ ]:
# Dataset path auto-detection: works on Kaggle and local workspace.
DATASET_CANDIDATES = [
    Path('/kaggle/input/deep-sar-oil-spill-segmentation-refined'),
    Path('/kaggle/input/datasets/bakhtiyar2222/deep-sar-oil-spill-segmentation-refined'),
    Path('/kaggle/input/archive-1'),
    Path('/kaggle/input/archive (1)'),
    Path('archive (1)'),
]
DATASET_ROOT = next((p for p in DATASET_CANDIDATES if (p / 'images' / 'images' / 'train').exists()), DATASET_CANDIDATES[0])

CONFIG = {
    "seed"            : 42,
    "model_name"      : "unetpp_efficientnetb4",
    "encoder_name"    : "efficientnet-b4",
    "encoder_weights" : "imagenet",
    "decoder_attention_type": "scse",
    "in_channels"     : 3,
    "classes"         : 1,
    "image_size"      : (256, 256),
    "threshold"       : 0.5,
    "threshold_search": [0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70],

    # Safer defaults for Kaggle T4/P100. Increase batch_size only if VRAM is stable.
    "batch_size"      : 8,
    "accumulation_steps": 2,
    "epochs"          : 120,
    "learning_rate"   : 8e-5,
    "weight_decay"    : 1e-4,
    "num_workers"     : 2,
    "early_stopping_patience" : 30,
    "grad_clip"       : 1.0,
    "mixed_precision" : True,

    # Recall was high and precision lower, so reduce pos_weight and add Tversky FP penalty.
    "loss_function"   : "BCE_Dice_Tversky",
    "bce_weight"      : 0.35,
    "dice_weight"     : 0.45,
    "tversky_weight"  : 0.20,
    "pos_weight"      : 2.0,
    "tversky_alpha"   : 0.70,
    "tversky_beta"    : 0.30,

    # SAR-friendly augmentation: geometric + mild noise/contrast only.
    "augmentation_prob": 0.35,

    "train_image_dir" : str(DATASET_ROOT / 'images' / 'images' / 'train'),
    "train_mask_dir"  : str(DATASET_ROOT / 'masks'  / 'masks'  / 'train'),
    "val_image_dir"   : str(DATASET_ROOT / 'images' / 'images' / 'val'),
    "val_mask_dir"    : str(DATASET_ROOT / 'masks'  / 'masks'  / 'val'),
    "checkpoint_dir"  : '/kaggle/working/checkpoints',
    "output_dir"      : '/kaggle/working/outputs',
}

Path(CONFIG['checkpoint_dir']).mkdir(parents=True, exist_ok=True)
Path(CONFIG['output_dir']).mkdir(parents=True, exist_ok=True)
(Path(CONFIG['output_dir']) / 'predictions').mkdir(parents=True, exist_ok=True)

print(f'DATASET_ROOT = {DATASET_ROOT}')
print('Dataset paths:')
for key in ['train_image_dir', 'train_mask_dir', 'val_image_dir', 'val_mask_dir']:
    p = Path(CONFIG[key])
    exists = p.exists()
    count  = len(list(p.glob('*.png'))) if exists else 0
    print(f'  {key:18s}: exists={exists} | n_png={count}')

print(f'\nimage_size = {CONFIG["image_size"]}')
print(f'threshold  = {CONFIG["threshold"]}')
print(f'batch_size = {CONFIG["batch_size"]} x accumulation {CONFIG["accumulation_steps"]}')
print(f'epochs     = {CONFIG["epochs"]}')
print(f'lr         = {CONFIG["learning_rate"]}')


## Cell 3. Fix seed

In [ ]:
def seed_everything(seed: int) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False
    print(f'✓ Seed cố định: {seed}')

seed_everything(CONFIG['seed'])
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'✓ Device: {DEVICE}')

## Cell 4. Load và EDA dataset

Kiểm tra cấu trúc thư mục, số lượng ảnh-mask, kích thước gốc, giá trị mask và mức độ mất cân bằng lớp.

**Kiểm tra cần đạt:**
```
SAR image shape: (416, 416, 3)
Mask shape:      (416, 416)
Mask values:     {0, 1} hoặc {0, 255}
```

In [ ]:
def load_image_mask_paths(image_dir: str, mask_dir: str) -> Tuple[List[Path], List[Path]]:
    image_paths = sorted(Path(image_dir).glob('*.png'))
    mask_paths  = sorted(Path(mask_dir).glob('*.png'))

    if not image_paths:
        raise FileNotFoundError(f'No PNG images found in: {image_dir}')
    if not mask_paths:
        raise FileNotFoundError(f'No PNG masks found in: {mask_dir}')

    image_names = {p.name for p in image_paths}
    mask_by_name = {p.name: p for p in mask_paths}
    missing = [p.name for p in image_paths if p.name not in mask_by_name]
    extra   = [p.name for p in mask_paths if p.name not in image_names]
    if missing:
        raise ValueError(f'{len(missing)} images have no matching mask. Example: {missing[:3]}')
    if extra:
        print(f'  WARNING: {len(extra)} masks have no matching image and will be ignored')

    paired_masks = [mask_by_name[p.name] for p in image_paths]
    print(f'  OK: paired {len(image_paths)} image-mask files by filename')
    return image_paths, paired_masks

print('=== Load train set ===')
train_imgs, train_masks = load_image_mask_paths(CONFIG['train_image_dir'], CONFIG['train_mask_dir'])
print('=== Load val set ===')
val_imgs, val_masks = load_image_mask_paths(CONFIG['val_image_dir'], CONFIG['val_mask_dir'])
print(f'\nTrain: {len(train_imgs)} images | Val: {len(val_imgs)} images')


In [ ]:
def compute_dataset_stats(image_paths: List[Path], n_sample: int = 100) -> dict:
    sample = image_paths[:n_sample]
    heights, widths, means = [], [], []
    for p in sample:
        img = cv2.imread(str(p))
        if img is not None:
            heights.append(img.shape[0])
            widths.append(img.shape[1])
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY).astype(np.float32)
            means.append(gray.mean())
    return {
        'height_min': int(np.min(heights)), 'height_max': int(np.max(heights)),
        'width_min' : int(np.min(widths)),  'width_max' : int(np.max(widths)),
        'n_channels': cv2.imread(str(image_paths[0])).shape[2] if image_paths else 0,
        'mean_intensity': float(np.mean(means)),
        'std_intensity' : float(np.std(means)),
    }

def compute_oil_ratios(mask_paths: List[Path], n_sample: int = 500) -> List[float]:
    ratios = []
    for p in mask_paths[:n_sample]:
        msk = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
        if msk is not None:
            ratios.append((msk > 127).sum() / msk.size)
    return ratios

print('Calculating dataset stats...')
stats      = compute_dataset_stats(train_imgs)
oil_ratios = compute_oil_ratios(train_masks)

print(f'\n── Kích thước ảnh gốc (train):')
print(f'   H: [{stats["height_min"]}, {stats["height_max"]}]  W: [{stats["width_min"]}, {stats["width_max"]}]')
print(f'   Số kênh: {stats["n_channels"]}')
print(f'\n── Pixel intensity:')
print(f'   Mean: {stats["mean_intensity"]:.1f}  |  Std: {stats["std_intensity"]:.1f}')
print(f'\n── Oil ratio (train, {len(oil_ratios)} mẫu):')
print(f'   Mean: {np.mean(oil_ratios)*100:.2f}%  |  Max: {np.max(oil_ratios)*100:.2f}%')
print(f'   Ảnh có dầu: {sum(r > 0 for r in oil_ratios)} / {len(oil_ratios)}')

test_mask = cv2.imread(str(train_masks[0]), cv2.IMREAD_GRAYSCALE)
print(f'\n── Giá trị mask (mẫu): {sorted(np.unique(test_mask).tolist())}')

In [ ]:
def show_eda_samples(img_paths, mask_paths, n=5, save_path=None):
    indices = random.sample(range(len(img_paths)), min(n, len(img_paths)))
    fig, axes = plt.subplots(3, len(indices), figsize=(4 * len(indices), 10))

    for col, idx in enumerate(indices):
        img_bgr = cv2.imread(str(img_paths[idx]))
        img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
        msk     = cv2.imread(str(mask_paths[idx]), cv2.IMREAD_GRAYSCALE)
        binary  = (msk > 127).astype(np.uint8)
        ratio   = binary.sum() / binary.size

        axes[0, col].imshow(img_rgb)
        axes[0, col].set_title(f'SAR\n{img_paths[idx].stem[:12]}', fontsize=8)
        axes[0, col].axis('off')

        axes[1, col].imshow(binary, cmap='Reds', vmin=0, vmax=1)
        axes[1, col].set_title(f'Mask\nratio={ratio*100:.1f}%', fontsize=8)
        axes[1, col].axis('off')

        overlay = img_rgb.copy()
        overlay[binary == 1] = [255, 80, 80]
        axes[2, col].imshow(overlay)
        axes[2, col].set_title('Overlay', fontsize=8)
        axes[2, col].axis('off')

    patch = mpatches.Patch(color='red', label='Oil spill')
    fig.legend(handles=[patch], loc='lower right', fontsize=10)
    fig.suptitle('Mẫu ảnh SAR — Mask Binary — Overlay', fontsize=14, fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
        print(f'  → Đã lưu: {save_path}')
    plt.show()

show_eda_samples(
    train_imgs, train_masks, n=5,
    save_path=str(Path(CONFIG['output_dir']) / 'eda_samples.png'),
)

In [ ]:
has_oil    = sum(1 for r in oil_ratios if r > 0)
no_oil     = len(oil_ratios) - has_oil
mean_ratio = np.mean(oil_ratios)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(oil_ratios, bins=40, color='#e05c5c', edgecolor='white', alpha=0.85)
axes[0].axvline(mean_ratio, color='navy', linestyle='--', linewidth=1.5,
                label=f'Mean = {mean_ratio*100:.2f}%')
axes[0].set_title('Phân bố tỷ lệ pixel dầu (training)', fontsize=12)
axes[0].set_xlabel('Tỷ lệ oil pixel')
axes[0].set_ylabel('Số ảnh')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

bars = axes[1].bar(
    ['Có dầu', 'Không dầu'], [has_oil, no_oil],
    color=['#e05c5c', '#4a90d9'], edgecolor='white', width=0.5
)
for bar, v in zip(bars, [has_oil, no_oil]):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.5,
        str(v), ha='center', fontweight='bold', fontsize=12,
    )
axes[1].set_title('Ảnh có / không có vùng dầu (training)', fontsize=12)
axes[1].set_ylabel('Số ảnh')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(str(Path(CONFIG['output_dir']) / 'class_distribution.png'), dpi=120)
plt.show()
print(f'Class imbalance (no_oil/has_oil): {no_oil / max(has_oil, 1):.2f}x')

## Cell 5. Pipeline tiền xử lý chung

Một pipeline Albumentations duy nhất được dùng cho **train, validation, test và inference**.

| Bước | Train | Val / Inference |
|---|:---:|:---:|
| Resize 256×256 | ✓ | ✓ |
| HorizontalFlip | ✓ | — |
| VerticalFlip | ✓ | — |
| RandomRotate90 | ✓ | — |
| ShiftScaleRotate | ✓ | — |
| RandomBrightnessContrast | ✓ | — |
| GaussNoise | ✓ | — |
| Mild speckle noise | ✓ | — |
| ElasticTransform + noise nhe | ✓ | — |
| Normalize (ImageNet) | ✓ | ✓ |
| ToTensorV2 | ✓ | ✓ |

In [ ]:
def build_transforms(is_train: bool = False) -> A.Compose:
    H, W = CONFIG['image_size']
    transforms = [A.Resize(height=H, width=W)]

    if is_train:
        prob = CONFIG['augmentation_prob']
        transforms.extend([
            A.HorizontalFlip(p=prob),
            A.VerticalFlip(p=prob),
            A.RandomRotate90(p=prob),
            A.ShiftScaleRotate(
                shift_limit=0.03,
                scale_limit=0.06,
                rotate_limit=8,
                border_mode=cv2.BORDER_CONSTANT,
                value=0,
                mask_value=0,
                p=prob,
            ),
            A.ElasticTransform(alpha=60, sigma=5, p=0.15),
            A.GaussNoise(var_limit=(8.0, 35.0), p=0.20),
            A.RandomBrightnessContrast(brightness_limit=0.12, contrast_limit=0.12, p=0.20),
        ])

    transforms.extend([
        A.Normalize(
            mean=(0.485, 0.456, 0.406),
            std =(0.229, 0.224, 0.225),
            max_pixel_value=255.0,
        ),
        ToTensorV2(),
    ])
    return A.Compose(transforms)

TRAIN_TRANSFORM     = build_transforms(is_train=True)
INFERENCE_TRANSFORM = build_transforms(is_train=False)

print('TRAIN_TRANSFORM:')
for t in TRAIN_TRANSFORM.transforms:
    print(f'  - {type(t).__name__}')
print('\nINFERENCE_TRANSFORM:')
for t in INFERENCE_TRANSFORM.transforms:
    print(f'  - {type(t).__name__}')


In [ ]:
def visualize_augmentations(img_paths, mask_paths, n_aug=4, save_path=None):
    idx     = random.randint(0, len(img_paths) - 1)
    img_bgr = cv2.imread(str(img_paths[idx]))
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    msk     = cv2.imread(str(mask_paths[idx]), cv2.IMREAD_GRAYSCALE)
    binary  = (msk > 127).astype(np.uint8)

    fig, axes = plt.subplots(2, n_aug + 1, figsize=(4 * (n_aug + 1), 7))

    axes[0, 0].imshow(img_rgb)
    axes[0, 0].set_title('Gốc (ảnh)', fontsize=9)
    axes[0, 0].axis('off')
    axes[1, 0].imshow(binary, cmap='Reds', vmin=0, vmax=1)
    axes[1, 0].set_title('Gốc (mask)', fontsize=9)
    axes[1, 0].axis('off')

    for i in range(n_aug):
        aug     = TRAIN_TRANSFORM(image=img_rgb, mask=binary)
        aug_img = aug['image'].permute(1, 2, 0).numpy()
        aug_img = (aug_img - aug_img.min()) / (aug_img.max() - aug_img.min() + 1e-8)
        aug_msk = aug['mask'].numpy()

        axes[0, i + 1].imshow(aug_img)
        axes[0, i + 1].set_title(f'Aug #{i+1} (ảnh)', fontsize=9)
        axes[0, i + 1].axis('off')
        axes[1, i + 1].imshow(aug_msk, cmap='Reds', vmin=0, vmax=1)
        axes[1, i + 1].set_title(f'Aug #{i+1} (mask)', fontsize=9)
        axes[1, i + 1].axis('off')

    fig.suptitle('Kiểm tra Augmentation — Ảnh SAR & Mask', fontsize=13, fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
        print(f'  → Đã lưu: {save_path}')
    plt.show()

visualize_augmentations(
    train_imgs, train_masks, n_aug=4,
    save_path=str(Path(CONFIG['output_dir']) / 'augmentation_samples.png'),
)

## Cell 6. Dataset class và DataLoader

In [ ]:
class OilSpillDataset(Dataset):
    def __init__(self, image_paths, mask_paths=None, transform=None):
        self.image_paths = image_paths
        self.mask_paths  = mask_paths
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        # Dataset images are RGB PNG, but SAR content is grayscale. Reading grayscale avoids channel surprises.
        img_gray = cv2.imread(str(self.image_paths[index]), cv2.IMREAD_GRAYSCALE)
        if img_gray is None:
            raise FileNotFoundError(f'Cannot read image: {self.image_paths[index]}')
        image = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2RGB)

        mask = None
        if self.mask_paths is not None:
            msk = cv2.imread(str(self.mask_paths[index]), cv2.IMREAD_GRAYSCALE)
            if msk is None:
                raise FileNotFoundError(f'Cannot read mask: {self.mask_paths[index]}')
            # Some train masks contain anti-aliased gray levels. Thresholding makes labels strictly binary.
            mask = (msk >= 128).astype(np.float32)

        if self.transform is not None:
            if mask is not None:
                transformed = self.transform(image=image, mask=mask)
                image = transformed['image']
                mask  = transformed['mask'].unsqueeze(0).float()
                mask  = (mask >= 0.5).float()
            else:
                image = self.transform(image=image)['image']

        if mask is None:
            return image
        return image, mask


train_dataset = OilSpillDataset(train_imgs, train_masks, transform=TRAIN_TRANSFORM)
val_dataset   = OilSpillDataset(val_imgs,   val_masks,   transform=INFERENCE_TRANSFORM)

train_loader = DataLoader(
    train_dataset, batch_size=CONFIG['batch_size'], shuffle=True,
    num_workers=CONFIG['num_workers'], pin_memory=torch.cuda.is_available(), drop_last=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=CONFIG['batch_size'], shuffle=False,
    num_workers=CONFIG['num_workers'], pin_memory=torch.cuda.is_available(),
)

print(f'Train dataset: {len(train_dataset)} samples ({len(train_loader)} batches)')
print(f'Val   dataset: {len(val_dataset)} samples ({len(val_loader)} batches)')

images, masks = next(iter(train_loader))
print(f'\nBatch images: {tuple(images.shape)}  (expected: Bx3x256x256)')
print(f'Batch masks : {tuple(masks.shape)}   (expected: Bx1x256x256)')
print(f'Mask unique : {torch.unique(masks).tolist()}')
assert images.shape[1:] == (3, CONFIG['image_size'][0], CONFIG['image_size'][1])
assert masks.shape[1:]  == (1, CONFIG['image_size'][0], CONFIG['image_size'][1])
print('Batch shape OK')


## Cell 7. Định nghĩa model UNet++

In [ ]:
model = smp.UnetPlusPlus(
    encoder_name    = CONFIG['encoder_name'],
    encoder_weights = CONFIG['encoder_weights'],
    decoder_attention_type = CONFIG['decoder_attention_type'],
    in_channels     = CONFIG['in_channels'],
    classes         = CONFIG['classes'],
    activation      = None,
).to(DEVICE)

dice_loss = smp.losses.DiceLoss(mode='binary', from_logits=True)
tversky_loss = smp.losses.TverskyLoss(
    mode='binary',
    from_logits=True,
    alpha=CONFIG['tversky_alpha'],
    beta=CONFIG['tversky_beta'],
)
bce_loss  = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([CONFIG['pos_weight']], device=DEVICE))

def combined_loss(logits, masks):
    bce = bce_loss(logits, masks)
    dice = dice_loss(logits, masks)
    tversky = tversky_loss(logits, masks)
    return (CONFIG['bce_weight'] * bce +
            CONFIG['dice_weight'] * dice +
            CONFIG['tversky_weight'] * tversky)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay'],
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CONFIG['epochs'],
    eta_min=1e-7,
)
scaler = GradScaler(enabled=CONFIG['mixed_precision'] and DEVICE.type == 'cuda')

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model     : UNet++ + {CONFIG["encoder_name"]}')
print(f'Attention : {CONFIG["decoder_attention_type"]}')
print(f'Total params    : {total_params:,}')
print(f'Trainable params: {trainable_params:,}')
print(f'AMP enabled     : {scaler.is_enabled()}')

with torch.no_grad():
    dummy = torch.randn(2, 3, CONFIG['image_size'][0], CONFIG['image_size'][1]).to(DEVICE)
    out   = model(dummy)
print(f'Forward: {tuple(dummy.shape)} -> {tuple(out.shape)}')


## Cell 8. Training loop

In [ ]:
def compute_binary_metrics_from_counts(tp, fp, fn, eps=1e-6):
    return {
        'iou': tp / (tp + fp + fn + eps),
        'dice': 2 * tp / (2 * tp + fp + fn + eps),
        'precision': tp / (tp + fp + eps),
        'recall': tp / (tp + fn + eps),
    }


def evaluate_with_threshold_search(model, loader, thresholds):
    model.eval()
    val_total = 0.0
    counts = {float(t): {'tp': 0.0, 'fp': 0.0, 'fn': 0.0} for t in thresholds}

    with torch.no_grad():
        for images_b, masks_b in loader:
            images_b = images_b.to(DEVICE, non_blocking=True)
            masks_b  = masks_b.to(DEVICE, non_blocking=True)
            with autocast(enabled=scaler.is_enabled()):
                logits = model(images_b)
                val_total += combined_loss(logits, masks_b).item()
            prob = torch.sigmoid(logits.float())
            gt = (masks_b > 0.5).float()

            for thr in counts:
                pred = (prob >= thr).float()
                counts[thr]['tp'] += (pred * gt).sum().item()
                counts[thr]['fp'] += (pred * (1 - gt)).sum().item()
                counts[thr]['fn'] += ((1 - pred) * gt).sum().item()

    best_thr = CONFIG['threshold']
    best_metrics = None
    for thr, c in counts.items():
        m = compute_binary_metrics_from_counts(c['tp'], c['fp'], c['fn'])
        if best_metrics is None or m['iou'] > best_metrics['iou']:
            best_thr = thr
            best_metrics = m

    return val_total / len(loader), float(best_thr), best_metrics


history  = []
best_iou = 0.0
best_threshold = CONFIG['threshold']
epochs_no_improve = 0
best_model_path = None
log_path = Path(CONFIG['output_dir']) / 'training_history.csv'

with open(log_path, 'w', newline='') as f:
    csv.writer(f).writerow(['epoch','train_loss','val_loss','iou','dice','precision','recall','threshold','lr'])

for epoch in range(1, CONFIG['epochs'] + 1):

    # Train
    model.train()
    train_total = 0.0
    optimizer.zero_grad(set_to_none=True)

    for step, (images_b, masks_b) in enumerate(tqdm(train_loader, desc=f'Epoch {epoch:03d}/{CONFIG["epochs"]} [Train]', leave=False), start=1):
        images_b = images_b.to(DEVICE, non_blocking=True)
        masks_b  = masks_b.to(DEVICE, non_blocking=True)

        with autocast(enabled=scaler.is_enabled()):
            logits = model(images_b)
            loss = combined_loss(logits, masks_b) / CONFIG['accumulation_steps']

        scaler.scale(loss).backward()

        if step % CONFIG['accumulation_steps'] == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        train_total += loss.item() * CONFIG['accumulation_steps']
    train_loss_avg = train_total / len(train_loader)

    # Validation with threshold tuning.
    val_loss_avg, val_thr, val_metrics = evaluate_with_threshold_search(
        model, val_loader, CONFIG['threshold_search']
    )
    val_iou = val_metrics['iou']
    val_dice = val_metrics['dice']
    val_prec = val_metrics['precision']
    val_rec = val_metrics['recall']

    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    row = dict(epoch=epoch, train_loss=train_loss_avg, val_loss=val_loss_avg,
               iou=val_iou, dice=val_dice, precision=val_prec, recall=val_rec,
               threshold=val_thr, lr=current_lr)
    history.append(row)
    with open(log_path, 'a', newline='') as f:
        csv.writer(f).writerow([epoch, f'{train_loss_avg:.6f}', f'{val_loss_avg:.6f}',
                                 f'{val_iou:.4f}', f'{val_dice:.4f}',
                                 f'{val_prec:.4f}', f'{val_rec:.4f}', f'{val_thr:.2f}', f'{current_lr:.8f}'])

    print(f'Epoch {epoch:03d}  LR={current_lr:.2e}  TrainLoss={train_loss_avg:.4f}  ValLoss={val_loss_avg:.4f}  '
          f'IoU={val_iou:.4f}  Dice={val_dice:.4f}  Prec={val_prec:.4f}  Rec={val_rec:.4f}  Thr={val_thr:.2f}')

    checkpoint_config = dict(CONFIG)
    checkpoint_config['threshold'] = float(val_thr)
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'val_iou': val_iou,
        'val_dice': val_dice,
        'val_precision': val_prec,
        'val_recall': val_rec,
        'threshold': float(val_thr),
        'config': checkpoint_config,
    }

    if val_iou > best_iou:
        old_best_path = best_model_path
        old_export_path = None
        if old_best_path is not None:
            old_export_path = Path('/kaggle/working') / old_best_path.name

        best_iou = val_iou
        best_threshold = val_thr
        epochs_no_improve = 0
        ckpt_name = f'unetpp_epoch{epoch:03d}_iou{val_iou:.4f}.pth'
        best_model_path = Path(CONFIG['checkpoint_dir']) / ckpt_name
        export_best_path = Path('/kaggle/working') / ckpt_name

        torch.save(checkpoint, best_model_path)
        shutil.copy2(best_model_path, export_best_path)

        if old_best_path is not None and old_best_path.exists() and old_best_path != best_model_path:
            old_best_path.unlink()
        if old_export_path is not None and old_export_path.exists() and old_export_path != export_best_path:
            old_export_path.unlink()

        print(f'  Best checkpoint: IoU={best_iou:.4f}, Thr={best_threshold:.2f} -> {best_model_path.name}')
        print(f'  Exported to /kaggle/working/{export_best_path.name}')
    else:
        epochs_no_improve += 1
        print(f'  No improvement since last best ({epochs_no_improve}/{CONFIG["early_stopping_patience"]})')
        if epochs_no_improve >= CONFIG['early_stopping_patience']:
            print(f'\nEarly stopping triggered after {epoch} epochs!')
            break

if best_model_path is None:
    raise RuntimeError('Training did not create a checkpoint. Check data, loss and metrics.')

CONFIG['threshold'] = float(best_threshold)
print(f'\nTraining done. Best IoU = {best_iou:.4f} at threshold {best_threshold:.2f}')
print(f'Best checkpoint: {best_model_path}')


## Cell 9. Visualize quá trình train

In [ ]:
df_hist = pd.DataFrame(history) if history else pd.read_csv(Path(CONFIG['output_dir']) / 'training_history.csv')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(df_hist['epoch'], df_hist['train_loss'], label='Train Loss', color='#e05c5c', linewidth=2)
ax1.plot(df_hist['epoch'], df_hist['val_loss'],   label='Val Loss',   color='#4a90d9', linewidth=2)
ax1.set_title('Loss theo Epoch', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.legend(); ax1.grid(True, alpha=0.3)

for col, color, label in [
    ('iou',       '#2ecc71', 'IoU'),
    ('dice',      '#3498db', 'Dice'),
    ('precision', '#9b59b6', 'Precision'),
    ('recall',    '#e67e22', 'Recall'),
]:
    ax2.plot(df_hist['epoch'], df_hist[col], label=label, color=color, linewidth=2)

best_row = df_hist.loc[df_hist['iou'].idxmax()]
ax2.axvline(best_row['epoch'], color='gray', linestyle='--', alpha=0.6,
            label=f'Best ep. {int(best_row["epoch"])}')
ax2.set_title('Metric theo Epoch', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Score'); ax2.set_ylim(0, 1.05)
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(str(Path(CONFIG['output_dir']) / 'loss_curve.png'),   dpi=120)
plt.savefig(str(Path(CONFIG['output_dir']) / 'metric_curve.png'), dpi=120)
plt.show()
print(f'Best ep {int(best_row["epoch"])}: IoU={best_row["iou"]:.4f}  Dice={best_row["dice"]:.4f}')

## Cell 10. Evaluation trên tập validation

In [ ]:
def resolve_checkpoint_path() -> Path:
    if 'best_model_path' in globals() and best_model_path is not None and Path(best_model_path).exists():
        return Path(best_model_path)
    ckpts = sorted(Path(CONFIG['checkpoint_dir']).glob('unetpp_epoch*_iou*.pth'))
    if not ckpts:
        raise FileNotFoundError(f'Khong tim thay checkpoint trong {CONFIG["checkpoint_dir"]}')
    return ckpts[-1]

best_model_path = resolve_checkpoint_path()

checkpoint = torch.load(best_model_path, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
CONFIG['threshold'] = float(checkpoint.get('threshold', checkpoint.get('config', {}).get('threshold', CONFIG['threshold'])))
print(f'Load: {best_model_path.name}  (epoch={checkpoint["epoch"]}, IoU={checkpoint["val_iou"]:.4f}, threshold={CONFIG["threshold"]:.2f})')

tp_e = fp_e = fn_e = val_loss_e = 0.0
with torch.no_grad():
    for images_b, masks_b in tqdm(val_loader, desc='Evaluation'):
        images_b = images_b.to(DEVICE, non_blocking=True)
        masks_b  = masks_b.to(DEVICE, non_blocking=True)
        with autocast(enabled=scaler.is_enabled()):
            logits = model(images_b)
            val_loss_e += combined_loss(logits, masks_b).item()
        prob  = torch.sigmoid(logits.float())
        pred  = (prob >= CONFIG['threshold']).float()
        gt    = (masks_b > 0.5).float()
        tp_e += (pred * gt).sum().item()
        fp_e += (pred * (1 - gt)).sum().item()
        fn_e += ((1 - pred) * gt).sum().item()

eps = 1e-6
final_metrics = {
    'model': 'UNet++', 'encoder': CONFIG['encoder_name'],
    'image_size': f'{CONFIG["image_size"][0]}x{CONFIG["image_size"][1]}',
    'threshold': CONFIG['threshold'],
    'val_loss' : val_loss_e / len(val_loader),
    'iou'      : tp_e / (tp_e + fp_e + fn_e + eps),
    'dice'     : 2 * tp_e / (2 * tp_e + fp_e + fn_e + eps),
    'precision': tp_e / (tp_e + fp_e + eps),
    'recall'   : tp_e / (tp_e + fn_e + eps),
}
with open(Path(CONFIG['output_dir']) / 'test_metrics.json', 'w') as f:
    json.dump(final_metrics, f, indent=2)

print('\nEvaluation result')
for k, v in final_metrics.items():
    if isinstance(v, float):
        print(f'  {k:12s}: {v:.4f}')
    else:
        print(f'  {k:12s}: {v}')


In [ ]:
def show_predictions(model, img_paths, mask_paths, n=4, save_path=None):
    indices = random.sample(range(len(img_paths)), min(n, len(img_paths)))
    fig, axes = plt.subplots(4, len(indices), figsize=(4 * len(indices), 14))
    row_labels = ['SAR Image', 'Ground Truth', 'Prediction', 'Overlay']

    model.eval()
    with torch.no_grad():
        for col, idx in enumerate(indices):
            img_gray = cv2.imread(str(img_paths[idx]), cv2.IMREAD_GRAYSCALE)
            img_rgb = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2RGB)
            msk = cv2.imread(str(mask_paths[idx]), cv2.IMREAD_GRAYSCALE)
            gt_bin = (msk >= 128).astype(np.uint8)

            tensor = INFERENCE_TRANSFORM(image=img_rgb)['image'].unsqueeze(0).to(DEVICE)
            prob = torch.sigmoid(model(tensor)).squeeze().cpu().numpy()
            pred = (prob >= CONFIG['threshold']).astype(np.uint8)

            gt_256  = cv2.resize(gt_bin,  CONFIG['image_size'][::-1], interpolation=cv2.INTER_NEAREST)
            img_256 = cv2.resize(img_rgb, CONFIG['image_size'][::-1], interpolation=cv2.INTER_LINEAR)
            overlay = img_256.copy()
            overlay[pred == 1] = [255, 80, 80]

            tp = (pred * gt_256).sum()
            iou_s = tp / (pred.sum() + gt_256.sum() - tp + 1e-6)

            for row, data, kw in [
                (0, img_256, {}),
                (1, gt_256,  dict(cmap='Reds', vmin=0, vmax=1)),
                (2, pred,    dict(cmap='Reds', vmin=0, vmax=1)),
                (3, overlay, {}),
            ]:
                axes[row, col].imshow(data, **kw)
                axes[row, col].axis('off')
                if row == 0:
                    axes[row, col].set_title(f'{img_paths[idx].stem[:12]}\nIoU={iou_s:.3f}', fontsize=8)

    for row, label in enumerate(row_labels):
        axes[row, 0].set_ylabel(label, fontsize=10, rotation=0, labelpad=60, va='center')

    fig.suptitle(f'Evaluation - SAR / GT / Pred / Overlay | threshold={CONFIG["threshold"]:.2f}', fontsize=13, fontweight='bold')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()

show_predictions(
    model, val_imgs, val_masks, n=4,
    save_path=str(Path(CONFIG['output_dir']) / 'test_predictions.png'),
)


## Cell 11. Inference với API thống nhất

API duy nhất `predict(image_path)` — không thêm bất kỳ tham số nào ngoài đường dẫn ảnh.

In [ ]:
def resolve_checkpoint_path() -> Path:
    if 'best_model_path' in globals() and best_model_path is not None and Path(best_model_path).exists():
        return Path(best_model_path)
    ckpts = sorted(Path(CONFIG['checkpoint_dir']).glob('unetpp_epoch*_iou*.pth'))
    if not ckpts:
        raise FileNotFoundError(f'Khong tim thay checkpoint trong {CONFIG["checkpoint_dir"]}')
    return ckpts[-1]

best_model_path = resolve_checkpoint_path()

checkpoint = torch.load(best_model_path, map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
CONFIG['threshold'] = float(checkpoint.get('threshold', checkpoint.get('config', {}).get('threshold', CONFIG['threshold'])))
print(f'Model ready: {best_model_path.name} | threshold={CONFIG["threshold"]:.2f}')


def predict(image_path) -> np.ndarray:
    """
    Return binary mask uint8 (256, 256) with 0=background and 1=oil spill.
    Unified API: only image_path.
    """
    img_gray = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if img_gray is None:
        raise FileNotFoundError(f'Cannot read image: {image_path}')
    img_rgb = cv2.cvtColor(img_gray, cv2.COLOR_GRAY2RGB)
    tensor  = INFERENCE_TRANSFORM(image=img_rgb)['image'].unsqueeze(0).to(DEVICE)
    model.eval()
    with torch.no_grad():
        probability = torch.sigmoid(model(tensor))
    return (probability.squeeze().cpu().numpy() >= CONFIG['threshold']).astype(np.uint8)


# Demo inference
demo_paths = random.sample(val_imgs, 3)
fig, axes  = plt.subplots(2, 3, figsize=(12, 7))

for col, img_path in enumerate(demo_paths):
    mask_pred = predict(img_path)
    img_gray = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)
    img_256 = cv2.resize(cv2.cvtColor(img_gray, cv2.COLOR_GRAY2RGB), CONFIG['image_size'][::-1])
    overlay = img_256.copy()
    overlay[mask_pred == 1] = [255, 80, 80]

    axes[0, col].imshow(img_256)
    axes[0, col].set_title(img_path.stem[:16], fontsize=8)
    axes[0, col].axis('off')
    axes[1, col].imshow(overlay)
    axes[1, col].set_title(f'oil px = {mask_pred.sum()}', fontsize=8)
    axes[1, col].axis('off')

fig.suptitle(f'Demo: predict(image_path), threshold={CONFIG["threshold"]:.2f}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(Path(CONFIG['output_dir']) / 'inference_demo.png'), dpi=120)
plt.show()

pred_dir = Path(CONFIG['output_dir']) / 'predictions'
for img_path in demo_paths:
    mask = predict(img_path)
    cv2.imwrite(str(pred_dir / f'{img_path.stem}_mask.png'), mask * 255)
print(f'Saved {len(demo_paths)} masks to {pred_dir}')


---

## Tong ket artifact

| Artifact | Duong dan |
|---|---|
| Checkpoint tot nhat, luu duy nhat | `/kaggle/working/checkpoints/unetpp_epoch{n}_iou{score}.pth` |
| Training history | `/kaggle/working/outputs/training_history.csv` |
| EDA samples | `/kaggle/working/outputs/eda_samples.png` |
| Class distribution | `/kaggle/working/outputs/class_distribution.png` |
| Augmentation samples | `/kaggle/working/outputs/augmentation_samples.png` |
| Loss curve | `/kaggle/working/outputs/loss_curve.png` |
| Metric curve | `/kaggle/working/outputs/metric_curve.png` |
| Evaluation metrics | `/kaggle/working/outputs/test_metrics.json` |
| Prediction samples | `/kaggle/working/outputs/test_predictions.png` |
| Inference demo | `/kaggle/working/outputs/inference_demo.png` |
| Prediction masks | `/kaggle/working/outputs/predictions/{stem}_mask.png` |


## Cell 12. Luu checkpoint thu cong

Chay cell nay sau training loop neu file `.pth` chua xuat hien trong `/kaggle/working`. Cell se luu model hien tai trong RAM thanh checkpoint va copy ra `/kaggle/working` de download tren Kaggle.


In [ ]:
from pathlib import Path
import torch
import shutil

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

if "model" not in globals():
    raise RuntimeError("Kernel khong con bien model. Neu da restart kernel thi phai train lai moi tao duoc checkpoint.")

iou_value = float(best_iou) if "best_iou" in globals() else 0.0
epoch_value = int(checkpoint["epoch"]) if "checkpoint" in globals() and isinstance(checkpoint, dict) else 0

ckpt_name = f"unetpp_epoch{epoch_value:03d}_iou{iou_value:.4f}.pth"
ckpt_path = CHECKPOINT_DIR / ckpt_name

torch.save({
    "epoch": epoch_value,
    "model_state_dict": model.state_dict(),
    "val_iou": iou_value,
    "threshold": float(CONFIG.get("threshold", 0.5)),
    "config": CONFIG,
    "note": "Manual rescue checkpoint saved after training.",
}, ckpt_path)

export_path = Path("/kaggle/working") / ckpt_name
shutil.copy2(ckpt_path, export_path)

print("Saved checkpoint:", ckpt_path)
print("Copied to:", export_path)
print("Size MB:", export_path.stat().st_size / 1024**2)
